# 02 Feature Engineering & Data Transformations

This notebook prepares PixelRec50K for recommender modeling. It covers:

1. Constructing user-item interaction samples and sparse matrices  
2. Encoding user IDs, item IDs, and item metadata  
3. Handling missing values, duplicates, and low-activity users/items  
4. Creating recommender features from item metadata  
5. Creating chronological train/validation/test splits without temporal leakage

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pickle
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer, StandardScaler
from scipy.sparse import csr_matrix, hstack, save_npz

Mounted at /content/drive


In [ ]:
DATASET_DIR = "/content/drive/MyDrive/PixelRec50K"
OUTPUT_DIR = os.path.join(DATASET_DIR, "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

interaction_path = os.path.join(DATASET_DIR, "pixel50k_interaction.csv")
item_info_path = os.path.join(DATASET_DIR, "pixel50k_item_info.csv")

interactions = pd.read_csv(interaction_path)
item_info = pd.read_csv(item_info_path)

interactions["datetime"] = pd.to_datetime(interactions["timestamp"], unit="s", errors="coerce")

print("Interactions:", interactions.shape)
print("Item info:", item_info.shape)
display(interactions.head())
display(item_info.head())

Interactions: (989494, 4)
Item info: (82865, 11)


,item_id,user_id,timestamp,datetime
0,i72138,u209296,1605059546,2020-11-11 01:52:26
1,i15530,u2444520,1628914341,2021-08-14 04:12:21
2,i95199,u1866870,1601008921,2020-09-25 04:42:01
3,i3413,u2498546,1505731122,2017-09-18 10:38:42
4,i224963,u3676118,1643894394,2022-02-03 13:19:54


,item_id,view_number,comment_number,thumbup_number,share_number,coin_number,favorite_number,barrage_number,title,tag,description
0,i192714,799668.0,739.0,8050.0,220.0,84.0,1049.0,510.0,"My boyfriend gave me a turtle, and it suddenly...",Pet Reptiles,Should I brush it off?
1,i225967,1201395.0,102.0,20424.0,738.0,502.0,4307.0,510.0,Who's not a baby anymore! Alaska is being bull...,Dogs,BGM: Don't Leave Me (Ne Me Quitte Pas) - Regin...
2,i8061,2633706.0,2502.0,160379.0,11161.0,2439.0,32801.0,1696.0,New York subway rules that pets can't ride the...,Dogs,wb
3,i217203,922049.0,244.0,9878.0,344.0,228.0,2129.0,1374.0,The biggest dog in the world. It's so ugly and...,Dogs,"Super love big dogs, the family has a baby Lab..."
4,i349133,186671.0,50.0,1630.0,67.0,87.0,1617.0,76.0,Taylor Swift: Selena and Mould perform Hands T...,Live Music,NaN


## 1. Data Cleaning

We remove invalid records, exact duplicates, and rows with missing user or item IDs. Since this is an implicit feedback dataset, each observed user-item interaction is treated as a positive signal.

In [ ]:
required_cols = ["user_id", "item_id", "timestamp"]

before = len(interactions)

interactions = interactions.dropna(subset=required_cols)
interactions = interactions.drop_duplicates(subset=["user_id", "item_id", "timestamp"])
interactions = interactions[interactions["datetime"].notna()]

after = len(interactions)

print(f"Rows before cleaning: {before:,}")
print(f"Rows after cleaning: {after:,}")
print(f"Rows removed: {before - after:,}")

print("\nMissing values after cleaning:")
print(interactions[required_cols].isna().sum())

Rows before cleaning: 989,494
Rows after cleaning: 989,494
Rows removed: 0

Missing values after cleaning:
user_id      0
item_id      0
timestamp    0
dtype: int64


## 2. Handle Low-Activity Users and Items

A 5-core filter is used to remove extremely sparse users/items. This improves collaborative filtering stability while retaining most interactions.

In [ ]:
def k_core_filter(df, min_user=5, min_item=5):
    filtered = df.copy()

    while True:
        start_len = len(filtered)

        user_counts = filtered["user_id"].value_counts()
        valid_users = user_counts[user_counts >= min_user].index
        filtered = filtered[filtered["user_id"].isin(valid_users)]

        item_counts = filtered["item_id"].value_counts()
        valid_items = item_counts[item_counts >= min_item].index
        filtered = filtered[filtered["item_id"].isin(valid_items)]

        if len(filtered) == start_len:
            break

    return filtered.reset_index(drop=True)

interactions_5core = k_core_filter(interactions, min_user=5, min_item=5)

print("Original interactions:", len(interactions))
print("5-core interactions:", len(interactions_5core))
print("Retention:", len(interactions_5core) / len(interactions))

print("Users:", interactions_5core["user_id"].nunique())
print("Items:", interactions_5core["item_id"].nunique())

Original interactions: 989494
5-core interactions: 913827
Retention: 0.9235296019985972
Users: 49950
Items: 47322


## 3. Encode User IDs and Item IDs

Raw user/item IDs are converted into integer indices so they can be used by matrix factorization, LightFM, Factorization Machines, and other recommender models.

In [ ]:
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

interactions_5core["user_idx"] = user_encoder.fit_transform(interactions_5core["user_id"])
interactions_5core["item_idx"] = item_encoder.fit_transform(interactions_5core["item_id"])

num_users = interactions_5core["user_idx"].nunique()
num_items = interactions_5core["item_idx"].nunique()

print("Encoded users:", num_users)
print("Encoded items:", num_items)

with open(os.path.join(OUTPUT_DIR, "user_encoder.pkl"), "wb") as f:
    pickle.dump(user_encoder, f)

with open(os.path.join(OUTPUT_DIR, "item_encoder.pkl"), "wb") as f:
    pickle.dump(item_encoder, f)

Encoded users: 49950
Encoded items: 47322


## 4. Chronological Train / Validation / Test Split

For each user, interactions are sorted by timestamp. Earlier interactions are used for training, middle interactions for validation, and latest interactions for testing. This avoids temporal leakage.

In [ ]:
def chronological_split(df, train_ratio=0.8, val_ratio=0.1):
    train_parts = []
    val_parts = []
    test_parts = []

    for _, user_df in df.sort_values("timestamp").groupby("user_idx"):
        n = len(user_df)

        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))

        if train_end < 1:
            train_end = 1
        if val_end <= train_end:
            val_end = train_end + 1
        if val_end >= n:
            val_end = n - 1

        train_parts.append(user_df.iloc[:train_end])
        val_parts.append(user_df.iloc[train_end:val_end])
        test_parts.append(user_df.iloc[val_end:])

    train_df = pd.concat(train_parts).reset_index(drop=True)
    val_df = pd.concat(val_parts).reset_index(drop=True)
    test_df = pd.concat(test_parts).reset_index(drop=True)

    return train_df, val_df, test_df

train_df, val_df, test_df = chronological_split(interactions_5core)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTime ranges:")
print("Train:", train_df["datetime"].min(), "to", train_df["datetime"].max())
print("Validation:", val_df["datetime"].min(), "to", val_df["datetime"].max())
print("Test:", test_df["datetime"].min(), "to", test_df["datetime"].max())

Train: (711060, 6)
Validation: (89743, 6)
Test: (113024, 6)

Time ranges:
Train: 2012-02-03 18:15:31 to 2022-06-14 09:26:03
Validation: 2016-09-26 14:32:18 to 2022-06-17 00:16:01
Test: 2016-12-18 03:56:37 to 2022-06-24 16:16:00


## 5. Construct User-Item Interaction Matrix

The training interactions are converted into a sparse user-item matrix. This matrix can be used for collaborative filtering models such as MF, NMF, implicit ALS, or LightFM.

In [ ]:
train_matrix = csr_matrix(
    (
        np.ones(len(train_df), dtype=np.float32),
        (train_df["user_idx"], train_df["item_idx"])
    ),
    shape=(num_users, num_items)
)

print("Train interaction matrix shape:", train_matrix.shape)
print("Non-zero entries:", train_matrix.nnz)

save_npz(os.path.join(OUTPUT_DIR, "train_interaction_matrix_5core.npz"), train_matrix)

Train interaction matrix shape: (49950, 47322)
Non-zero entries: 711060


## 6. Negative Sampling

For implicit feedback models that require labeled samples, observed interactions are treated as positive examples and randomly sampled unobserved items are treated as negative examples.

In [ ]:
rng = np.random.default_rng(42)

all_items = np.arange(num_items)

user_positive_items = (
    interactions_5core
    .groupby("user_idx")["item_idx"]
    .apply(set)
    .to_dict()
)

def sample_negative_items(user_idx, n_negatives):
    positives = user_positive_items[user_idx]
    negatives = []

    while len(negatives) < n_negatives:
        candidate = rng.integers(0, num_items)
        if candidate not in positives:
            negatives.append(candidate)

    return negatives

def make_labeled_samples(pos_df, negatives_per_positive=1):
    positive_samples = pos_df[["user_idx", "item_idx"]].copy()
    positive_samples["label"] = 1

    negative_rows = []

    for row in pos_df[["user_idx"]].itertuples(index=False):
        user_idx = row.user_idx
        neg_items = sample_negative_items(user_idx, negatives_per_positive)

        for item_idx in neg_items:
            negative_rows.append((user_idx, item_idx, 0))

    negative_samples = pd.DataFrame(
        negative_rows,
        columns=["user_idx", "item_idx", "label"]
    )

    samples = pd.concat([positive_samples, negative_samples], ignore_index=True)
    samples = samples.sample(frac=1, random_state=42).reset_index(drop=True)

    return samples

train_samples = make_labeled_samples(train_df, negatives_per_positive=1)
val_samples = make_labeled_samples(val_df, negatives_per_positive=5)
test_samples = make_labeled_samples(test_df, negatives_per_positive=5)

print("Train samples:", train_samples.shape)
print("Validation samples:", val_samples.shape)
print("Test samples:", test_samples.shape)
display(train_samples.head())

Train samples: (1422120, 3)
Validation samples: (538458, 3)
Test samples: (678144, 3)


,user_idx,item_idx,label
0,19495,23534,0
1,21385,28764,0
2,44596,10423,0
3,4394,6388,0
4,42477,5673,0


## 7. Item Metadata Feature Engineering

Item metadata is used to create recommender features. Tags are encoded as categorical features, and numeric engagement fields are log-transformed and standardized.

Note: engagement counts such as views, likes, and favorites may contain popularity information aggregated over time. They should be used carefully because they may introduce temporal leakage.

In [ ]:
item_info_clean = item_info.copy()
item_info_clean.columns = item_info_clean.columns.str.strip()

valid_item_ids = set(item_encoder.classes_)
item_info_clean = item_info_clean[item_info_clean["item_id"].isin(valid_item_ids)].copy()

item_info_clean["item_idx"] = item_encoder.transform(item_info_clean["item_id"])

text_cols = ["title", "tag", "description"]
for col in text_cols:
    if col in item_info_clean.columns:
        item_info_clean[col] = item_info_clean[col].fillna("Missing").astype(str)

numeric_cols = [
    "view_number",
    "comment_number",
    "thumbup_number",
    "share_number",
    "coin_number",
    "favorite_number",
    "barrage_number"
]

available_numeric_cols = [c for c in numeric_cols if c in item_info_clean.columns]

for col in available_numeric_cols:
    item_info_clean[col] = item_info_clean[col].fillna(0)
    item_info_clean[f"log_{col}"] = np.log1p(item_info_clean[col])

log_numeric_cols = [f"log_{c}" for c in available_numeric_cols]

print("Metadata items after filtering:", item_info_clean.shape)
print("Numeric feature columns:", log_numeric_cols)
display(item_info_clean.head())

Metadata items after filtering: (47322, 19)
Numeric feature columns: ['log_view_number', 'log_comment_number', 'log_thumbup_number', 'log_share_number', 'log_coin_number', 'log_favorite_number', 'log_barrage_number']


,item_id,view_number,comment_number,thumbup_number,share_number,coin_number,favorite_number,barrage_number,title,tag,description,item_idx,log_view_number,log_comment_number,log_thumbup_number,log_share_number,log_coin_number,log_favorite_number,log_barrage_number
0,i192714,799668.0,739.0,8050.0,220.0,84.0,1049.0,510.0,"My boyfriend gave me a turtle, and it suddenly...",Pet Reptiles,Should I brush it off?,17798,13.591953,6.606650,8.993552,5.398163,4.442651,6.956545,6.236370
2,i8061,2633706.0,2502.0,160379.0,11161.0,2439.0,32801.0,1696.0,New York subway rules that pets can't ride the...,Dogs,wb,43508,14.783903,7.825245,11.985301,9.320270,7.799753,10.398245,7.436617
3,i217203,922049.0,244.0,9878.0,344.0,228.0,2129.0,1374.0,The biggest dog in the world. It's so ugly and...,Dogs,"Super love big dogs, the family has a baby Lab...",22638,13.734355,5.501258,9.198167,5.843544,5.433722,7.663877,7.226209
6,i349130,205967.0,292.0,14972.0,1576.0,1591.0,3827.0,329.0,Feng Mo DIO VS Chong Huang JOJO,Miscellaneous,Missing,33220,12.235476,5.680173,9.614004,7.363280,7.372746,8.250098,5.799093
7,i349124,478510.0,293.0,31098.0,1664.0,4515.0,11761.0,767.0,DIO's bad adventures] JOJO invincible edition,Short Film·Hand-drawn·Dubbing,"A precious video footage, taken from a paralle...",33219,13.078434,5.683580,10.344931,7.417580,8.415382,9.372629,6.643790


In [ ]:
# Encode tags as multi-hot categorical features
item_info_clean["tag_list"] = (
    item_info_clean["tag"]
    .fillna("Missing")
    .astype(str)
    .str.split(",")
    .apply(lambda tags: [t.strip() for t in tags if t.strip()])
)

mlb = MultiLabelBinarizer()
tag_features = mlb.fit_transform(item_info_clean["tag_list"])

tag_feature_df = pd.DataFrame(
    tag_features,
    columns=[f"tag_{t}" for t in mlb.classes_]
)

print("Number of tag features:", tag_feature_df.shape[1])
display(tag_feature_df.head())

Number of tag features: 115


,tag_Agriculture,tag_Animals Mix,tag_Animation and Radio Drama,tag_Basketball,tag_Basketball and Football,tag_Beauty and Skincare,tag_Board Games and Card Games,tag_Campus Learning,tag_Car Buying Guide,tag_Car Culture,...,tag_Tutorials and Demonstrations,tag_VOCALOID·UTAU,tag_Variety Shows,tag_Western Films,tag_Wild Skills Association,tag_Wildlife,tag_and Farmers,tag_and Nature,tag_and Psychology,tag_and Travel
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Numeric metadata features
scaler = StandardScaler()

if log_numeric_cols:
    numeric_features = scaler.fit_transform(item_info_clean[log_numeric_cols])
    numeric_feature_df = pd.DataFrame(
        numeric_features,
        columns=[f"scaled_{c}" for c in log_numeric_cols]
    )
else:
    numeric_feature_df = pd.DataFrame(index=item_info_clean.index)

display(numeric_feature_df.head())

,scaled_log_view_number,scaled_log_comment_number,scaled_log_thumbup_number,scaled_log_share_number,scaled_log_coin_number,scaled_log_favorite_number,scaled_log_barrage_number
0,0.443800,0.071596,-0.538653,-0.826311,-1.562623,-0.984692,-0.478130
1,1.527280,1.219331,1.663535,1.448061,0.086478,1.348530,0.428814
2,0.573242,-0.969518,-0.388039,-0.568041,-1.075782,-0.505172,0.269823
3,-0.789236,-0.801008,-0.081946,0.313231,-0.123279,-0.107757,-0.808550
4,-0.022988,-0.797799,0.456080,0.344719,0.388892,0.653238,-0.170271


In [ ]:
# Simple text-length features from title and description
item_info_clean["title_length"] = item_info_clean["title"].fillna("").str.len()
item_info_clean["description_length"] = item_info_clean["description"].fillna("").str.len()

length_features = scaler.fit_transform(
    item_info_clean[["title_length", "description_length"]]
)

length_feature_df = pd.DataFrame(
    length_features,
    columns=["scaled_title_length", "scaled_description_length"]
)

display(length_feature_df.head())

,scaled_title_length,scaled_description_length
0,0.468629,-0.670061
1,0.108897,-0.768584
2,-0.370745,0.950647
3,-1.162156,-0.743954
4,-0.826406,0.300393


In [ ]:
item_feature_df = pd.concat(
    [
        item_info_clean[["item_id", "item_idx"]].reset_index(drop=True),
        tag_feature_df.reset_index(drop=True),
        numeric_feature_df.reset_index(drop=True),
        length_feature_df.reset_index(drop=True)
    ],
    axis=1
)

item_feature_df = item_feature_df.sort_values("item_idx").reset_index(drop=True)

print("Item feature table:", item_feature_df.shape)
display(item_feature_df.head())

item_feature_df.to_csv(
    os.path.join(OUTPUT_DIR, "item_features_5core.csv"),
    index=False
)

with open(os.path.join(OUTPUT_DIR, "tag_mlb.pkl"), "wb") as f:
    pickle.dump(mlb, f)

with open(os.path.join(OUTPUT_DIR, "metadata_scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

Item feature table: (47322, 126)


,item_id,item_idx,tag_Agriculture,tag_Animals Mix,tag_Animation and Radio Drama,tag_Basketball,tag_Basketball and Football,tag_Beauty and Skincare,tag_Board Games and Card Games,tag_Campus Learning,...,tag_and Travel,scaled_log_view_number,scaled_log_comment_number,scaled_log_thumbup_number,scaled_log_share_number,scaled_log_coin_number,scaled_log_favorite_number,scaled_log_barrage_number,scaled_title_length,scaled_description_length
0,i10,0,0,0,0,0,0,0,0,0,...,0,0.062368,0.239064,0.717776,0.086685,1.164943,0.851987,0.635884,-1.377995,0.536849
1,i1000,1,0,0,0,0,0,0,0,0,...,0,1.278329,0.511082,1.042776,2.233236,1.189335,2.147137,0.335572,0.108897,1.310257
2,i100001,2,0,0,0,0,0,0,0,0,...,0,0.328848,-0.614933,-0.144186,0.151898,-0.549890,0.694167,0.124211,-0.058978,-0.596169
3,i100015,3,0,0,0,0,0,0,0,0,...,0,0.624456,1.345239,0.944497,1.955931,1.364081,1.333801,1.586946,-0.154906,2.265933
4,i100022,4,0,0,0,0,0,0,0,0,...,0,1.066494,0.180832,0.398755,-0.070258,-0.332429,0.408434,1.126781,2.579057,0.664929


## 8. Save Processed Training Files

The processed files are saved for downstream model notebooks. All models should use these files to ensure fair comparison.

In [ ]:
keep_cols = [
    "user_id",
    "item_id",
    "user_idx",
    "item_idx",
    "timestamp",
    "datetime"
]

train_df[keep_cols].to_csv(os.path.join(OUTPUT_DIR, "train_5core.csv"), index=False)
val_df[keep_cols].to_csv(os.path.join(OUTPUT_DIR, "val_5core.csv"), index=False)
test_df[keep_cols].to_csv(os.path.join(OUTPUT_DIR, "test_5core.csv"), index=False)

train_samples.to_csv(os.path.join(OUTPUT_DIR, "train_samples_5core.csv"), index=False)
val_samples.to_csv(os.path.join(OUTPUT_DIR, "val_samples_5core.csv"), index=False)
test_samples.to_csv(os.path.join(OUTPUT_DIR, "test_samples_5core.csv"), index=False)

interactions_5core.to_csv(os.path.join(OUTPUT_DIR, "interactions_clean_5core.csv"), index=False)

print("Saved processed files to:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

Saved processed files to: /content/drive/MyDrive/PixelRec50K/processed
['user_encoder.pkl', 'item_encoder.pkl', 'train_interaction_matrix_5core.npz', 'item_features_5core.csv', 'tag_mlb.pkl', 'metadata_scaler.pkl', 'train_5core.csv', 'val_5core.csv', 'test_5core.csv', 'train_samples_5core.csv', 'val_samples_5core.csv', 'test_samples_5core.csv', 'interactions_clean_5core.csv']


## 9. Sanity Checks

In [ ]:
print("=== Split Sizes ===")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\n=== Encoded ID Ranges ===")
print("User idx range:", interactions_5core["user_idx"].min(), interactions_5core["user_idx"].max())
print("Item idx range:", interactions_5core["item_idx"].min(), interactions_5core["item_idx"].max())

print("\n=== Sample Label Distribution ===")
print("Train:")
print(train_samples["label"].value_counts(normalize=True))
print("Validation:")
print(val_samples["label"].value_counts(normalize=True))
print("Test:")
print(test_samples["label"].value_counts(normalize=True))

print("\n=== Item Feature Alignment ===")
print("Feature rows:", len(item_feature_df))
print("Unique item_idx:", item_feature_df["item_idx"].nunique())
print("Expected num_items:", num_items)

=== Split Sizes ===
Train: 711060
Validation: 89743
Test: 113024

=== Encoded ID Ranges ===
User idx range: 0 49949
Item idx range: 0 47321

=== Sample Label Distribution ===
Train:
label
0    0.5
1    0.5
Name: proportion, dtype: float64
Validation:
label
0    0.833333
1    0.166667
Name: proportion, dtype: float64
Test:
label
0    0.833333
1    0.166667
Name: proportion, dtype: float64

=== Item Feature Alignment ===
Feature rows: 47322
Unique item_idx: 47322
Expected num_items: 47322


In [ ]:
# Check temporal ordering per user
split_check = []

for user_idx in interactions_5core["user_idx"].unique()[:1000]:
    train_max = train_df.loc[train_df["user_idx"] == user_idx, "timestamp"].max()
    val_min = val_df.loc[val_df["user_idx"] == user_idx, "timestamp"].min()
    test_min = test_df.loc[test_df["user_idx"] == user_idx, "timestamp"].min()

    split_check.append({
        "user_idx": user_idx,
        "train_before_val": train_max <= val_min if pd.notna(val_min) else True,
        "val_before_test": val_min <= test_min if pd.notna(test_min) else True
    })

split_check_df = pd.DataFrame(split_check)

print("Train before validation:", split_check_df["train_before_val"].mean())
print("Validation before test:", split_check_df["val_before_test"].mean())
display(split_check_df.head())

Train before validation: 1.0
Validation before test: 1.0


,user_idx,train_before_val,val_before_test
0,6781,True,True
1,12460,True,True
2,2689,True,True
3,13265,True,True
4,27314,True,True
